# Proyecto II – Detección de anomalías con MVTec ADEste notebook guía la implementación solicitada en el enunciado universitario.Se estructura por secciones claramente etiquetadas para que el profesor puedaver qué requisito se cumple en cada bloque.

## 1. Introducción y setup- Objetivo: construir sistemas de detección de anomalías usando MVTec AD.- Tecnologías: PyTorch Lightning, Hydra, ResNet-18, distillation teacher–student,  autoencoder U-Net, Mahalanobis, PCA/t-SNE, DBSCAN.

In [ ]:
# Instalación de dependencias en Google Colab# (esta celda se omite si ya se tienen los paquetes instalados)# !pip install -q pytorch-lightning torchvision hydra-core scikit-learn matplotlib seaborn

In [ ]:
# Clonar el repositorio si se trabaja directamente en Colab# from google.colab import drive# drive.mount('/content/drive')# !git clone https://github.com/usuario/ProyectoIO2Test.git

## 2. Carga de configuración con HydraEn esta sección se demuestra el uso de `conf/config.yaml` y los subdirectorios deHydra tal como exige el enunciado.

In [ ]:
import hydrafrom omegaconf import OmegaConf# Cargamos la configuración principalcfg = OmegaConf.load('conf/config.yaml')print(OmegaConf.to_yaml(cfg))

## 3. Preparación de datos (LightningDataModule)Usamos el `MVTecDataModule` para cargar únicamente ejemplos **normales** en elentrenamiento, cumpliendo la restricción del enunciado.

In [ ]:
from hydra.utils import instantiatefrom src.data.datamodule import MVTecDataModule# Instanciamos el DataModule desde la configuracióntrain_tfms = instantiate(cfg.data.train_transforms)test_tfms = instantiate(cfg.data.test_transforms)datamodule = MVTecDataModule(    root=cfg.data.root,    category=cfg.data.category,    batch_size=cfg.data.batch_size,    num_workers=cfg.data.num_workers,    train_transforms=train_tfms,    test_transforms=test_tfms,)datamodule.setup()print(f"Tamaño train: {len(datamodule.mvtec_train)} | val/test: {len(datamodule.mvtec_val)}")

## 4. Entrenamiento de modelosA continuación se muestran bloques separados para cada modelo solicitado en elenunciado.

### 4.A Modelo A – ResNet-18 scratch (Sección III.A)En esta sección entrenamos el modelo A (ResNet-18 scratch) como se indica en la sección III.A del enunciado.

In [ ]:
import pytorch_lightning as plfrom pytorch_lightning.callbacks import EarlyStopping# Instanciamos modelo y entrenador desde Hydramodel_a = instantiate(cfg.model)trainer_cfg = instantiate(cfg.trainer)logger_cfg = instantiate(cfg.logger)trainer_a = trainer_cfgtrainer_a.callbacks = [EarlyStopping(**cfg.early_stopping)]trainer_a.logger = logger_cfg# Entrenamiento (comentar para evitar ejecuciones accidentales)# trainer_a.fit(model_a, datamodule=datamodule)# trainer_a.test(model_a, dataloaders=datamodule)

### 4.B Modelo B – Distillation teacher–student (Sección III.B)Se entrena al estudiante ligero guiado por el teacher ResNet-18.

In [ ]:
# Seleccionamos la configuración específica del modelo Bcfg_b = OmegaConf.load('conf/model/resnet_distillation.yaml')model_b = instantiate(cfg_b)trainer_b = instantiate(cfg.trainer)trainer_b.callbacks = [EarlyStopping(**cfg.early_stopping)]trainer_b.logger = instantiate(cfg.logger)# Entrenamiento del student# trainer_b.fit(model_b, datamodule=datamodule)

### 4.C Modelo C – Autoencoder U-Net (Sección III.C)En esta sección entrenamos el autoencoder U-Net solicitado.

In [ ]:
cfg_c = OmegaConf.load('conf/model/unet_autoencoder.yaml')model_c = instantiate(cfg_c)trainer_c = instantiate(cfg.trainer)trainer_c.callbacks = [EarlyStopping(**cfg.early_stopping)]trainer_c.logger = instantiate(cfg.logger)# Entrenamiento del autoencoder# trainer_c.fit(model_c, datamodule=datamodule)

## 5. Cálculo de embeddings y distancia de Mahalanobis (Sección IV)
Comparativa completa para los tres modelos: extraemos embeddings, estimamos (μ, Σ) con ejemplos normales y calculamos la distancia de Mahalanobis en validación y prueba.

In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_recall_curve, auc, precision_score, recall_score, f1_score, confusion_matrix

from src.utils.embeddings_eval import score_samples, mahalanobis_distance, compute_gaussian_stats

# Utilidad genérica para extraer embeddings y etiquetas
def compute_embeddings(model, dataloader, device):
    # Recorre un DataLoader y acumula embeddings + etiquetas en CPU.
    model.eval()
    all_embeddings, all_labels = [], []
    with torch.no_grad():
        for x, y in dataloader:
            x = x.to(device)
            emb = model.extract_embeddings(x)
            all_embeddings.append(emb.cpu())
            all_labels.append(y)
    return torch.cat(all_embeddings), torch.cat(all_labels)


# Métricas a partir de distancias y un umbral dado
def summarize_distances(dists, labels, threshold):
    labels_np = labels.numpy()
    dists_np = dists.numpy()
    preds = (dists_np > threshold).astype(int)
    precision = precision_score(labels_np, preds, zero_division=0)
    recall = recall_score(labels_np, preds, zero_division=0)
    f1 = f1_score(labels_np, preds, zero_division=0)
    rc, pr, thr = precision_recall_curve(labels_np, dists_np)
    pr_auc = auc(rc, pr)
    cm = confusion_matrix(labels_np, preds)
    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "threshold": threshold,
        "precision_recall": (rc, pr),
        "pr_auc": pr_auc,
        "confusion_matrix": cm,
        "preds": preds,
    }


# Pipeline completo de Mahalanobis para un modelo dado
def evaluate_mahalanobis_for_model(model, loaders, device, threshold_quantile: float = 0.95):
    # Calcula (μ, Σ), distancias y métricas en val/test para un modelo.
    # - loaders: tupla (train_loader, val_loader, test_loader)
    # - threshold_quantile: percentil tomado sobre distancias de validación normales
    train_loader, val_loader, test_loader = loaders
    model = model.to(device)

    # Estadísticos gaussianos usando solo entrenamiento (imágenes normales)
    train_embeddings, train_labels = compute_embeddings(model, train_loader, device)
    mu, cov = compute_gaussian_stats(train_embeddings)

    def _process_split(loader):
        embeddings, labels = compute_embeddings(model, loader, device)
        dists = mahalanobis_distance(embeddings, mu, cov)
        return {"embeddings": embeddings, "labels": labels, "dists": dists}

    val_data = _process_split(val_loader)
    normal_mask = val_data["labels"] == 0
    val_threshold = torch.quantile(val_data["dists"][normal_mask], threshold_quantile).item()

    test_data = _process_split(test_loader)

    return {
        "train": {"embeddings": train_embeddings, "labels": train_labels, "mu": mu, "cov": cov},
        "val": {**val_data, "metrics": summarize_distances(val_data["dists"], val_data["labels"], val_threshold)},
        "test": {**test_data, "metrics": summarize_distances(test_data["dists"], test_data["labels"], val_threshold)},
    }


In [ ]:
# Ejecutamos el cálculo de embeddings y distancias Mahalanobis para A/B/C
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
loaders = (datamodule.train_dataloader(), datamodule.val_dataloader(), datamodule.test_dataloader())

models_info = [
    {"key": "A", "name": "ResNet-18 scratch", "model": model_a},
    {"key": "B", "name": "ResNet-18 distillation", "model": model_b},
    {"key": "C", "name": "Autoencoder U-Net", "model": model_c},
]

mahalanobis_results = {}
for info in models_info:
    print(f"Procesando {info['name']}...")
    mahalanobis_results[info['key']] = evaluate_mahalanobis_for_model(info['model'], loaders, device)

# Resumen tabular de métricas en validación y prueba
rows = []
for info in models_info:
    key = info['key']
    rows.append({
        'Modelo': info['name'],
        'F1 val': mahalanobis_results[key]['val']['metrics']['f1'],
        'Precisión val': mahalanobis_results[key]['val']['metrics']['precision'],
        'Recall val': mahalanobis_results[key]['val']['metrics']['recall'],
        'Umbral val': mahalanobis_results[key]['val']['metrics']['threshold'],
        'F1 test': mahalanobis_results[key]['test']['metrics']['f1'],
        'Precisión test': mahalanobis_results[key]['test']['metrics']['precision'],
        'Recall test': mahalanobis_results[key]['test']['metrics']['recall'],
    })
mahalanobis_summary = pd.DataFrame(rows)
mahalanobis_summary


## 6. Reducción de dimensionalidad con PCA y t-SNE (Sección V)
Aplicamos PCA y t-SNE a los embeddings de validación de cada modelo para comparar la estructura de sus espacios latentes.

In [ ]:
from src.utils.dim_reduction import apply_pca, apply_tsne

reduction_results = {}
for info in models_info:
    key = info['key']
    emb = mahalanobis_results[key]['val']['embeddings']
    labels = mahalanobis_results[key]['val']['labels']
    pca_2d = apply_pca(emb, n_components=2)
    tsne_2d = apply_tsne(emb, n_components=2, perplexity=cfg.embeddings.tsne_perplexity)
    reduction_results[key] = {"labels": labels, "pca": pca_2d, "tsne": tsne_2d}

fig, axes = plt.subplots(len(models_info), 2, figsize=(12, 4*len(models_info)))
if len(models_info) == 1:
    axes = np.array([axes])
for idx, info in enumerate(models_info):
    key = info['key']
    sns.scatterplot(x=reduction_results[key]['pca'][:,0], y=reduction_results[key]['pca'][:,1], hue=reduction_results[key]['labels'], ax=axes[idx,0], palette='Set1', s=30)
    axes[idx,0].set_title(f"PCA 2D - {info['name']}")
    sns.scatterplot(x=reduction_results[key]['tsne'][:,0], y=reduction_results[key]['tsne'][:,1], hue=reduction_results[key]['labels'], ax=axes[idx,1], palette='Set1', s=30)
    axes[idx,1].set_title(f"t-SNE 2D - {info['name']}")
plt.tight_layout()
plt.show()

# Comparativa de distancias Mahalanobis y curvas PR
fig, ax = plt.subplots(figsize=(8, 4))
for info in models_info:
    key = info['key']
    sns.kdeplot(mahalanobis_results[key]['val']['dists'].numpy(), ax=ax, label=info['name'])
ax.set_title('Distribución de distancias Mahalanobis (val)')
ax.set_xlabel('Distancia')
plt.legend()
plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
for info in models_info:
    key = info['key']
    rc, pr = mahalanobis_results[key]['val']['metrics']['precision_recall']
    ax.plot(rc, pr, label=f"{info['name']} (AUC={mahalanobis_results[key]['val']['metrics']['pr_auc']:.3f})")
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Curvas Precision-Recall (val)')
ax.legend()
plt.show()


## 7. Clustering de outliers con DBSCAN (Sección VI)
Evaluamos DBSCAN sobre las proyecciones PCA de cada modelo y comparamos cuántos puntos marcados como ruido coinciden con anomalías.

In [ ]:
from src.utils.clustering import apply_dbscan

dbscan_results = {}
dbscan_rows = []
for info in models_info:
    key = info['key']
    labels = reduction_results[key]['labels']
    pca_space = reduction_results[key]['pca']
    clustering = apply_dbscan(pca_space, eps=cfg.embeddings.dbscan.eps, min_samples=cfg.embeddings.dbscan.min_samples)
    noise = clustering['noise_mask']
    anomalies = labels == 1
    detected_anomalies = (noise & anomalies).sum().item()
    false_noise = (noise & (~anomalies)).sum().item()
    dbscan_results[key] = {
        'labels': clustering['labels'],
        'noise_mask': noise,
        'n_clusters': clustering['n_clusters'],
        'detected_anomalies': detected_anomalies,
        'false_noise': false_noise,
    }
    dbscan_rows.append({
        'Modelo': info['name'],
        'Clusters': clustering['n_clusters'],
        'Ruido total': int(noise.sum().item()),
        'Anomalías detectadas como ruido': detected_anomalies,
        'Normales marcadas como ruido': false_noise,
    })

fig, axes = plt.subplots(1, len(models_info), figsize=(5*len(models_info), 4))
if len(models_info) == 1:
    axes = [axes]
for ax, info in zip(axes, models_info):
    key = info['key']
    sns.scatterplot(x=reduction_results[key]['pca'][:,0], y=reduction_results[key]['pca'][:,1], hue=dbscan_results[key]['labels'], palette='tab10', ax=ax, s=25)
    ax.set_title(f"DBSCAN (PCA) - {info['name']}
Clusters={dbscan_results[key]['n_clusters']}")
plt.tight_layout()
plt.show()

dbscan_summary = pd.DataFrame(dbscan_rows)
dbscan_summary


## 8. Análisis de resultados
Resumen automático de métricas, distancias y comportamiento de clustering para comparar los tres modelos.

In [ ]:
# Tabla consolidada de métricas de anomalía (Mahalanobis)
analysis_table = mahalanobis_summary.copy()
analysis_table['AUC PR val'] = [mahalanobis_results[info['key']]['val']['metrics']['pr_auc'] for info in models_info]
analysis_table['AUC PR test'] = [mahalanobis_results[info['key']]['test']['metrics']['pr_auc'] for info in models_info]
print('Resumen de métricas por modelo:')
display(analysis_table)

print('
Resumen DBSCAN por modelo:')
display(dbscan_summary)

# Comentarios programáticos basados en los resultados
for info in models_info:
    key = info['key']
    val_f1 = mahalanobis_results[key]['val']['metrics']['f1']
    test_f1 = mahalanobis_results[key]['test']['metrics']['f1']
    auc_val = mahalanobis_results[key]['val']['metrics']['pr_auc']
    print(f"
[{info['name']}] F1 val={val_f1:.3f} | F1 test={test_f1:.3f} | AUC PR val={auc_val:.3f}")
    print(f"DBSCAN → clusters: {dbscan_results[key]['n_clusters']} | ruido total: {dbscan_results[key]['noise_mask'].sum().item()} | anomalías en ruido: {dbscan_results[key]['detected_anomalies']}")


## 9. Checklist de cumplimiento del enunciadoLista rápida de verificación final.

In [ ]:
checklist = {    'Hydra con config.yaml y subdirectorios': '✅',    'Entrenamiento solo con datos normales': '✅',    'LightningModule para cada modelo (A/B/C)': '✅',    'LightningDataModule para MVTec AD': '✅',    'Callbacks de EarlyStopping configurados': '✅',    'Evaluación con embeddings y Mahalanobis': '✅',    'PCA y t-SNE para visualización': '✅',    'DBSCAN para outliers': '✅',}for k, v in checklist.items():    print(f"{v} {k}")